# 01: Structured Global Climate Mesh (NetCDF-4)

This notebook demonstrates 3D spherical visualization of structured rectilinear climate data with **pyearthviz3d** and **geovista**:
1. Loading latitude/longitude coordinates and scalar fields (temperature and elevation) from CF-compliant NetCDF-4 (`data/structured/structured_global_climate.nc`).
2. Constructing a 3D spherical mesh using `geovista.Transform.from_1d`.
3. Configuring camera vantage point, coastlines, graticule, and horizontal colorbar with `VisualizationConfig` and `ScalarBarConfig`.
4. Rendering high-resolution 3D visualizations and displaying them inline.

In [ ]:
import os
from pathlib import Path
import netCDF4 as nc
import numpy as np
import geovista as gv
from IPython.display import Image, display

import pyearthviz3d
from pyearthviz3d import (
    VisualizationConfig,
    ScalarBarConfig,
    map_single_frame,
)

# Resolve repository root and data path
REPO_ROOT = Path(os.getcwd()).resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATA_PATH = REPO_ROOT / "data" / "structured" / "structured_global_climate.nc"
print(f"pyearthviz3d version: {pyearthviz3d.__version__}")
print(f"Target NetCDF dataset: {DATA_PATH} (exists: {DATA_PATH.exists()})")

## Step 1: Load and Inspect the Structured Climate Dataset

Let's read the NetCDF dataset containing global 2-meter air temperature and surface elevation.

In [ ]:
with nc.Dataset(DATA_PATH, "r") as ds:
    lats = ds.variables["lat"][:]
    lons = ds.variables["lon"][:]
    temp = ds.variables["temperature"][:]
    elev = ds.variables["elevation"][:]
    attrs = {var: ds.variables[var].units for var in ["temperature", "elevation"]}

print(f"Grid dimensions: {len(lats)} latitudes x {len(lons)} longitudes")
print(f"Latitude range:  [{lats.min():.1f}°, {lats.max():.1f}°]")
print(f"Longitude range: [{lons.min():.1f}°, {lons.max():.1f}°]")
print(f"Temperature: min = {temp.min():.1f} K, max = {temp.max():.1f} K")
print(f"Elevation:   min = {elev.min():.1f} m, max = {elev.max():.1f} m")

## Step 2: Construct the 3D Spherical Mesh

We use `geovista.Transform.from_1d(lons, lats)` to transform 1D coordinate vectors into a 3D spherical mesh wrapped onto the Earth's radius, and attach cell-averaged scalar values.

In [ ]:
# Construct 3D spherical mesh from 1D coordinates
mesh = gv.Transform.from_1d(lons, lats)

# Compute cell-centered scalar values from corner nodes
temp_cells = (temp[:-1, :-1] + temp[1:, :-1] + temp[:-1, 1:] + temp[1:, 1:]) / 4.0
mesh.cell_data["temperature"] = temp_cells.ravel()

elev_cells = (elev[:-1, :-1] + elev[1:, :-1] + elev[:-1, 1:] + elev[1:, 1:]) / 4.0
mesh.cell_data["elevation"] = elev_cells.ravel()

valid_indices = np.arange(mesh.n_cells)
print(f"Constructed PyVista/GeoVista mesh: {mesh.n_cells} cells, {mesh.n_points} points.")

## Step 3: Render Global Surface Temperature

We configure camera focus over the Americas (`longitude_focus=-95.0`, `latitude_focus=35.0`), enable geographic coastlines and graticule lines, and attach a formatted horizontal scalar bar.

In [ ]:
viz_config = VisualizationConfig(
    longitude_focus=-95.0,  # Centered over North America
    latitude_focus=35.0,
    zoom_factor=0.85,
    show_coastlines=True,
    show_graticule=True,
    coastline_color="black",
    coastline_width=1.2,
    colormap="plasma",
    verbose=False,
)

scalar_config = ScalarBarConfig(
    title="Surface Air Temperature",
    title_font_size=12,
    label_font_size=10,
    fmt="%.1f",
    n_labels=6,
    orientation="horizontal",
)

output_png = "structured_mesh_temperature.png"
result = map_single_frame(
    pMesh=mesh,
    aValid_cell_indices=valid_indices,
    pConfig=viz_config,
    sScalar="temperature",
    sUnit="K",
    sFilename_out=output_png,
    scalar_config=scalar_config,
    show_edges=False,
    return_detailed_result=True,
)

print(result.get_summary())
display(Image(output_png, width=650))

## Step 4: Render Global Topographic Elevation

We now re-render the same mesh with the `elevation` scalar field, shifting our camera vantage point to East Asia / Himalayas (`longitude_focus=85.0`, `latitude_focus=28.0`) and using the `terrain` colormap.

In [ ]:
viz_elev_config = VisualizationConfig(
    longitude_focus=85.0,  # Focus over Himalayas / East Asia
    latitude_focus=28.0,
    zoom_factor=0.9,
    show_coastlines=True,
    show_graticule=True,
    coastline_color="white",
    coastline_width=1.0,
    colormap="terrain",
    verbose=False,
)

scalar_elev_config = ScalarBarConfig(
    title="Topographic Elevation",
    title_font_size=12,
    label_font_size=10,
    fmt="%.0f",
    n_labels=5,
    orientation="horizontal",
)

output_elev_png = "structured_mesh_elevation.png"
map_single_frame(
    pMesh=mesh,
    aValid_cell_indices=valid_indices,
    pConfig=viz_elev_config,
    sScalar="elevation",
    sUnit="m",
    sFilename_out=output_elev_png,
    scalar_config=scalar_elev_config,
    show_edges=False,
)

display(Image(output_elev_png, width=650))